# Homework 01 — Home Price Prediction using Prompt Engineering

## Objective
Use **Prompt Engineering with Few-Shot Examples** to predict California home prices.

### Tools
- Google Colab
- Python / Pandas
- LangChain
- OpenAI
- Colab Secrets

### Dataset
`california_housing_train.csv`

### Approach
1. Load the California housing dataset.
2. Randomly select **65 rows** as few-shot examples.
3. Convert those examples into a structured prompt.
4. Use LangChain + OpenAI to predict `median_house_value` for a new house.
5. Evaluate the prompt-based prediction on held-out rows.

> This notebook does **not** train a traditional ML regression model. The goal is to use an LLM with in-context/few-shot examples.


## 1. Install Required Libraries

Run this cell once in Google Colab.


In [14]:
!pip install -q langchain langchain-openai pandas

## 2. Import Libraries and Read OpenAI API Key from Colab Secrets

In Google Colab:
1. Click the **key icon** in the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Enable notebook access for that secret.


In [15]:
import os
import re
import random
import pandas as pd
import numpy as np

from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add it in Google Colab Secrets."
    )

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI API key loaded successfully.")


OpenAI API key loaded successfully.


## 3. Upload and Load the California Housing Dataset

Upload `california_housing_train.csv` to the Colab session before running this cell.


In [16]:
DATASET_PATH = "/content/sample_data/california_housing_train.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (17000, 9)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


## 4. Inspect the Dataset

The target variable for prediction is:

`median_house_value`


In [17]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe())


Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           17000 non-null  float64
 1   latitude            17000 non-null  float64
 2   housing_median_age  17000 non-null  float64
 3   total_rooms         17000 non-null  float64
 4   total_bedrooms      17000 non-null  float64
 5   population          17000 non-null  float64
 6   households          17000 non-null  float64
 7   median_income       17000 non-null  float64
 8   median_house_value  17000 non-null  float64
dtypes: float64(9)
memory usage: 1.2 MB

Missing values:


,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,0
population,0
households,0
median_income,0
median_house_value,0



Summary statistics:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000
mean,-119.562108,35.625225,28.589353,2643.664412,539.410824,1429.573941,501.221941,3.883578,207300.912353
std,2.005166,2.137340,12.586937,2179.947071,421.499452,1147.852959,384.520841,1.908157,115983.764387
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.790000,33.930000,18.000000,1462.000000,297.000000,790.000000,282.000000,2.566375,119400.000000
50%,-118.490000,34.250000,29.000000,2127.000000,434.000000,1167.000000,409.000000,3.544600,180400.000000
75%,-118.000000,37.720000,37.000000,3151.250000,648.250000,1721.000000,605.250000,4.767000,265000.000000
max,-114.310000,41.950000,52.000000,37937.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


## 5. Randomly Select 65 Rows for Few-Shot Prompt Examples

The homework asks for approximately **60–70 random rows**.  
This notebook uses **65 rows**, which satisfies that requirement.

A fixed `random_state` is used so the notebook remains reproducible.


In [18]:
NUM_FEW_SHOT_EXAMPLES = 65
RANDOM_STATE = 42

# Keep the original dataset indices so we can exclude these exact rows
# from later evaluation.
few_shot_df = df.sample(
    n=NUM_FEW_SHOT_EXAMPLES,
    random_state=RANDOM_STATE
).copy()

few_shot_indices = few_shot_df.index

print(f"Selected {len(few_shot_df)} random rows.")
display(few_shot_df.head())


Selected 65 random rows.


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
10941,-120.87,37.77,9.0,4838.0,920.0,2460.0,923.0,3.5959,142700.0
5250,-118.14,34.11,52.0,2742.0,422.0,1153.0,414.0,8.1124,500001.0
10292,-120.05,36.98,16.0,3705.0,739.0,2463.0,697.0,2.5288,61800.0
2266,-117.42,34.02,9.0,5455.0,882.0,3015.0,858.0,4.2321,162800.0
6398,-118.26,33.97,52.0,1331.0,346.0,1144.0,362.0,1.5326,90600.0


## 6. Convert the Random Rows into Few-Shot Examples

Each row contains the housing attributes followed by the known target value.  
These examples are inserted into the prompt so the LLM can infer the relationship between the input features and house prices.


In [19]:
FEATURE_COLUMNS = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
]

TARGET_COLUMN = "median_house_value"


def format_house(row, include_target=True):
    lines = [
        f"Longitude: {row['longitude']}",
        f"Latitude: {row['latitude']}",
        f"Housing Median Age: {row['housing_median_age']}",
        f"Total Rooms: {row['total_rooms']}",
        f"Total Bedrooms: {row['total_bedrooms']}",
        f"Population: {row['population']}",
        f"Households: {row['households']}",
        f"Median Income: {row['median_income']}",
    ]

    if include_target:
        lines.append(
            f"Median House Value: ${row['median_house_value']:,.2f}"
        )

    return "\n".join(lines)


def build_few_shot_examples(example_df):
    examples = []

    for index, row in example_df.iterrows():
        example_text = (
            f"Example {index + 1}:\n"
            f"{format_house(row, include_target=True)}"
        )
        examples.append(example_text)

    return "\n\n".join(examples)


few_shot_examples = build_few_shot_examples(few_shot_df)

print(few_shot_examples[:2500])
print("\n... prompt examples truncated for display ...")


Example 10942:
Longitude: -120.87
Latitude: 37.77
Housing Median Age: 9.0
Total Rooms: 4838.0
Total Bedrooms: 920.0
Population: 2460.0
Households: 923.0
Median Income: 3.5959
Median House Value: $142,700.00

Example 5251:
Longitude: -118.14
Latitude: 34.11
Housing Median Age: 52.0
Total Rooms: 2742.0
Total Bedrooms: 422.0
Population: 1153.0
Households: 414.0
Median Income: 8.1124
Median House Value: $500,001.00

Example 10293:
Longitude: -120.05
Latitude: 36.98
Housing Median Age: 16.0
Total Rooms: 3705.0
Total Bedrooms: 739.0
Population: 2463.0
Households: 697.0
Median Income: 2.5288
Median House Value: $61,800.00

Example 2267:
Longitude: -117.42
Latitude: 34.02
Housing Median Age: 9.0
Total Rooms: 5455.0
Total Bedrooms: 882.0
Population: 3015.0
Households: 858.0
Median Income: 4.2321
Median House Value: $162,800.00

Example 6399:
Longitude: -118.26
Latitude: 33.97
Housing Median Age: 52.0
Total Rooms: 1331.0
Total Bedrooms: 346.0
Population: 1144.0
Households: 362.0
Median Income: 1

## 7. Create the Prompt Template

The prompt tells the LLM to:
- study the historical examples,
- infer relationships between housing attributes and price,
- estimate the value for a new house,
- return a structured response that can be parsed reliably.


In [20]:
PROMPT_TEMPLATE = '''
You are an AI assistant performing California home price prediction.

Your task is to estimate the median house value for a new California housing
record by studying the labeled historical examples below.

Important instructions:
1. Treat the historical rows as few-shot examples.
2. Consider geographic location, housing age, number of rooms and bedrooms,
   population, households, and median income.
3. Use patterns across all provided examples rather than relying on only one row.
4. The target variable is median_house_value.
5. Return one estimated value in US dollars.
6. Do not return a range.
7. Do not include commas inside the numeric value.
8. Use exactly this final format:

PREDICTED_PRICE: <number>

Historical few-shot examples:
{few_shot_examples}

New house:
{new_house}

Predict the median house value.
'''

prompt_template = PromptTemplate(
    input_variables=["few_shot_examples", "new_house"],
    template=PROMPT_TEMPLATE
)

print("Prompt template created.")


Prompt template created.


## 8. Initialize the OpenAI Model using LangChain

`temperature=0` is used to make predictions more deterministic.

If your OpenAI account does not support the model below, replace it with another chat model available to your account.


In [21]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("OpenAI model initialized through LangChain.")


OpenAI model initialized through LangChain.


## 9. Create the Prediction Function

This function:
1. formats the new house,
2. builds the final few-shot prompt,
3. sends it to OpenAI,
4. extracts the numeric prediction.


In [22]:
def extract_prediction(response_text):
    match = re.search(
        r"PREDICTED_PRICE\s*:\s*\$?([0-9]+(?:\.[0-9]+)?)",
        response_text,
        flags=re.IGNORECASE,
    )

    if not match:
        raise ValueError(
            f"Could not extract prediction from model response:\n{response_text}"
        )

    return float(match.group(1))


def predict_house_price(house_attributes, verbose=False):
    house_series = pd.Series(house_attributes)

    new_house_text = format_house(
        house_series,
        include_target=False
    )

    final_prompt = prompt_template.format(
        few_shot_examples=few_shot_examples,
        new_house=new_house_text
    )

    if verbose:
        print("PROMPT PREVIEW")
        print("=" * 80)
        print(final_prompt[:5000])
        print("\n... prompt truncated for display ...")
        print("=" * 80)

    response = llm.invoke(final_prompt)
    prediction = extract_prediction(response.content)

    return prediction, response.content


print("Prediction function ready.")


Prediction function ready.


## 10. Predict the Price for a New House

Change the values below to test a new property.

The fields must match the dataset attributes.


In [23]:
new_house = {
    "longitude": -118.25,
    "latitude": 34.05,
    "housing_median_age": 30.0,
    "total_rooms": 2200.0,
    "total_bedrooms": 450.0,
    "population": 1200.0,
    "households": 420.0,
    "median_income": 5.5,
}

prediction, raw_response = predict_house_price(
    new_house,
    verbose=False
)

print("Model response:")
print(raw_response)

print(f"\nPredicted Median House Value: ${prediction:,.2f}")


Model response:
The new house is located at Longitude -118.25 and Latitude 34.05, which is very close to several examples in the dataset around the 34 latitude and -118 longitude area.

Key features of the new house:
- Housing Median Age: 30.0
- Total Rooms: 2200.0
- Total Bedrooms: 450.0
- Population: 1200.0
- Households: 420.0
- Median Income: 5.5

Let's compare with similar examples:

1. Example 7913:
   - Longitude: -118.4, Latitude: 33.93
   - Age: 35.0
   - Rooms: 2217
   - Bedrooms: 447
   - Population: 1000
   - Households: 450
   - Median Income: 4.7319
   - Median House Value: $376,100

2. Example 6551:
   - Longitude: -118.28, Latitude: 34.24
   - Age: 29.0
   - Rooms: 3390
   - Bedrooms: 580
   - Population: 1543
   - Households: 576
   - Median Income: 5.6184
   - Median House Value: $316,900

3. Example 8199:
   - Longitude: -118.44, Latitude: 34.24
   - Age: 35.0
   - Rooms: 2344
   - Bedrooms: 435
   - Population: 1531
   - Households: 399
   - Median Income: 3.725
   -

## 11. Optional Interactive User Input

Run this cell if you want a user to manually enter house attributes.


In [24]:
def get_user_house_input():
    return {
        "longitude": float(input("Longitude: ")),
        "latitude": float(input("Latitude: ")),
        "housing_median_age": float(input("Housing median age: ")),
        "total_rooms": float(input("Total rooms: ")),
        "total_bedrooms": float(input("Total bedrooms: ")),
        "population": float(input("Population: ")),
        "households": float(input("Households: ")),
        "median_income": float(input("Median income: ")),
    }


# Uncomment the following lines when you want interactive input.

# user_house = get_user_house_input()
# user_prediction, user_raw_response = predict_house_price(user_house)
# print(f"Predicted Median House Value: ${user_prediction:,.2f}")


## 12. Evaluate the Prompt on Held-Out Dataset Rows

To make the homework stronger, we can evaluate whether the LLM's predictions are reasonably close to known values.

The rows used for evaluation are excluded from the 65 few-shot examples.


In [25]:
# Exclude all 65 rows that were already shown to the LLM.
remaining_df = df.drop(index=few_shot_indices)

test_df = remaining_df.sample(
    n=5,
    random_state=101
).reset_index(drop=True)

print("Few-shot rows:", len(few_shot_df))
print("Remaining unseen rows:", len(remaining_df))
print("Evaluation rows:", len(test_df))

display(test_df)


Few-shot rows: 65
Remaining unseen rows: 16935
Evaluation rows: 5


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-117.65,34.10,44.0,1526.0,337.0,831.0,326.0,3.0284,115800.0
1,-119.74,34.43,26.0,3119.0,562.0,1459.0,562.0,5.0434,340400.0
2,-121.06,37.86,24.0,1713.0,328.0,1258.0,324.0,2.6830,169400.0
3,-117.94,33.82,34.0,1347.0,212.0,676.0,201.0,3.8828,215400.0
4,-120.02,38.76,15.0,3142.0,618.0,725.0,285.0,4.3333,121400.0


In [26]:
evaluation_results = []

for i, row in test_df.iterrows():
    input_features = {
        column: row[column]
        for column in FEATURE_COLUMNS
    }

    actual_price = float(row[TARGET_COLUMN])

    predicted_price, _ = predict_house_price(
        input_features,
        verbose=False
    )

    absolute_error = abs(actual_price - predicted_price)

    percentage_error = (
        absolute_error / actual_price * 100
        if actual_price != 0
        else np.nan
    )

    evaluation_results.append({
        "test_case": i + 1,
        "actual_price": actual_price,
        "predicted_price": predicted_price,
        "absolute_error": absolute_error,
        "percentage_error": percentage_error,
    })


results_df = pd.DataFrame(evaluation_results)

display(
    results_df.style.format({
        "actual_price": "${:,.2f}",
        "predicted_price": "${:,.2f}",
        "absolute_error": "${:,.2f}",
        "percentage_error": "{:.2f}%"
    })
)


,test_case,actual_price,predicted_price,absolute_error,percentage_error
0,1,"$115,800.00","$170,000.00","$54,200.00",46.80%
1,2,"$340,400.00","$360,000.00","$19,600.00",5.76%
2,3,"$169,400.00","$135,000.00","$34,400.00",20.31%
3,4,"$215,400.00","$180,000.00","$35,400.00",16.43%
4,5,"$121,400.00","$230,000.00","$108,600.00",89.46%


## 13. Summarize Evaluation Metrics

**Mean Absolute Error (MAE)** shows the average dollar difference between the actual and predicted prices.

**Mean Absolute Percentage Error (MAPE)** shows the average error as a percentage of the actual value.

These metrics are used only to evaluate the prompt-based predictions.  
No traditional ML model is trained in this homework.


In [27]:
mae = results_df["absolute_error"].mean()
mape = results_df["percentage_error"].mean()

print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")


Mean Absolute Error (MAE): $50,440.00
Mean Absolute Percentage Error (MAPE): 35.75%


## 14. Conclusion

This homework demonstrates how a large language model can perform a numeric prediction task through **few-shot prompt engineering**.

### What was done
- Loaded the California housing dataset using Pandas.
- Randomly sampled **65 labeled housing records**.
- Converted those records into structured few-shot examples.
- Built a reusable prompt with LangChain's `PromptTemplate`.
- Called an OpenAI chat model through LangChain.
- Predicted `median_house_value` for unseen housing attributes.
- Compared prompt-based predictions with known prices from held-out rows.

### Important Observation
This technique is an example of **in-context learning**, not conventional machine-learning model training. The LLM receives examples inside its context window and uses those examples to infer an answer for a new record.

Because LLMs are general-purpose language models rather than specialized regression models, a conventional trained regression model may achieve better numerical accuracy. However, this exercise demonstrates how structured prompting and few-shot examples can adapt a general-purpose LLM to a predictive task without model training.


## 15. Recommended GitHub Files

A clean homework repository can contain:

```text
Homework_01_Home_Price_Prediction/
│
├── Homework_01_Home_Price_Prediction.ipynb
├── california_housing_train.csv
└── README.md
```

### Security
Never commit your OpenAI API key to GitHub.  
Keep it inside **Google Colab Secrets** as `OPENAI_API_KEY`.
